# S3_02_Regresion_Logistica.ipynb (Versión 7.0 Definitiva)
**Basado estrictamente en la guía de la Parte 2 (`guia_cesar_parte2.md`)**
- **Celda 1:** Carga de datos de Fernanda, recálculo de edad de ingreso con `anio_ing_carr_act`, variables derivadas, X e Y.
- **Celda 2:** 3 modelos (M1, M2, Stepwise), evaluación por AIC/BIC.
- **Celda 3:** Bootstrap (B=1000, semilla 42, grupos por nomb_inst), CIs e iteraciones.
- **Celda 4:** Evaluación en prueba con umbral Youden de entrenamiento, métricas, matriz de confusión y ROC.
- **Celda 5:** Exportación de `logistica_modelos.csv`, `logistica_odds_ratios.csv` y `logistica_metricas.csv`.

In [ ]:
# CELDA 1: Carga de datos (parte 1 de Fernanda) y preparación de variables
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# 1. Datos de la parte 1: imputación por regresión (E3), partición 70/30 estratificada, semilla 42
train = pd.read_parquet("../resultados/train_e3.parquet")
test = pd.read_parquet("../resultados/test_e3.parquet")

# 2. Variables derivadas
def preparar(d):
    d = d.copy()
    d["sexo"] = d["gen_alu"].map({1: "Hombre", 2: "Mujer"})
    # Se usa anio_ing_carr_act porque anio_ing_carr_ori queda vacío en los casos imputados
    d["edad_ingreso"] = d["anio_ing_carr_act"] - d["fec_nac_alu"] // 100
    return d

train = preparar(train)
test = preparar(test)

# 3. Variable objetivo: atraso alto (atraso > 4 semestres, definido en la S1)
y_train = train["atraso_alto"].astype(int)
y_test = test["atraso_alto"].astype(int)

# 4. Predictores: sexo, área y jornada (S1/S2) más la edad de ingreso
predictoras = ["sexo", "area_conocimiento", "jornada", "edad_ingreso"]
print("Nulos en predictoras (debe ser 0):", int(train[predictoras].isna().sum().sum()))

X_train = sm.add_constant(pd.get_dummies(train[predictoras], drop_first=True, dtype=int))
X_test = sm.add_constant(pd.get_dummies(test[predictoras], drop_first=True, dtype=int), has_constant="add")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)   # mismas columnas que en entrenamiento

print(f"Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}")
print(f"% atraso alto -> entrenamiento: {y_train.mean()*100:.2f} % | prueba: {y_test.mean()*100:.2f} %")
print(train["edad_ingreso"].describe().round(1))

In [ ]:
# CELDA 2: Tres modelos y comparación por AIC/BIC

def columnas(variables):
    """Columnas de X (dummies incluidas) que corresponden a una lista de variables."""
    return ["const"] + [c for c in X_train.columns
                        if any(c == v or c.startswith(v + "_") for v in variables)]

def ajustar(variables):
    return sm.Logit(y_train, X_train[columnas(variables)]).fit(disp=False)

# Modelo 1: variables de S1/S2
vars_m1 = ["sexo", "area_conocimiento", "jornada"]
m1 = ajustar(vars_m1)

# Modelo 2: S1/S2 + edad de ingreso
vars_m2 = vars_m1 + ["edad_ingreso"]
m2 = ajustar(vars_m2)

# Modelo 3: stepwise hacia adelante por AIC. Entra o sale la variable completa
# (todas sus dummies juntas), no dummy por dummy.
def stepwise_aic(candidatas):
    elegidas = []
    aic_actual = sm.Logit(y_train, X_train[["const"]]).fit(disp=False).aic
    while True:
        pruebas = {v: ajustar(elegidas + [v]).aic for v in candidatas if v not in elegidas}
        if not pruebas:
            break
        mejor = min(pruebas, key=pruebas.get)
        if pruebas[mejor] < aic_actual:
            elegidas.append(mejor)
            print(f"  entra {mejor:<20} AIC = {pruebas[mejor]:.1f}")
            aic_actual = pruebas[mejor]
        else:
            break
    return elegidas

print("Stepwise:")
vars_m3 = stepwise_aic(predictoras)
m3 = ajustar(vars_m3)

modelos = {"M1 · S1/S2": (m1, vars_m1), "M2 · S1/S2 + edad": (m2, vars_m2), "M3 · stepwise": (m3, vars_m3)}
tabla_modelos = pd.DataFrame([
    {"Modelo": k, "Variables": ", ".join(v), "Parámetros": len(m.params),
     "AIC": round(m.aic, 1), "BIC": round(m.bic, 1), "Pseudo R²": round(m.prsquared, 4)}
    for k, (m, v) in modelos.items()]).set_index("Modelo")
print(tabla_modelos)

# Se elige el de menor BIC (penaliza más la complejidad); si coincide con el de menor AIC, mejor
nombre_mejor = tabla_modelos["BIC"].idxmin()
mejor_modelo, vars_mejor = modelos[nombre_mejor]
cols_mejor = columnas(vars_mejor)
print(f"\nMenor AIC: {tabla_modelos['AIC'].idxmin()} | Menor BIC: {nombre_mejor}")
print(f"Modelo seleccionado: {nombre_mejor}")
print(mejor_modelo.summary())

In [ ]:
# CELDA 3: Bootstrap de odds ratios remuestreando universidades (nomb_inst)
B = 1000
rng = np.random.default_rng(42)

grupos = train.groupby("nomb_inst").indices     # posiciones de cada universidad
instituciones = list(grupos)
Xv = X_train[cols_mejor].to_numpy(float)
yv = y_train.to_numpy()

coefs, fallidas = [], 0
for b in range(B):
    elegidas = rng.choice(len(instituciones), size=len(instituciones), replace=True)
    idx = np.concatenate([grupos[instituciones[i]] for i in elegidas])
    try:
        r = sm.Logit(yv[idx], Xv[idx]).fit(disp=False)
        if not r.mle_retvals["converged"]:
            raise ValueError("no convergió")
        coefs.append(r.params)
    except Exception:
        fallidas += 1

coefs = np.array(coefs)
print(f"Universidades: {len(instituciones)} | B = {B} | exitosas: {len(coefs)} | fallidas: {fallidas}")

wald = mejor_modelo.conf_int()
tabla_or = pd.DataFrame({
    "OR": np.exp(mejor_modelo.params),
    "IC95 bootstrap inf": np.exp(np.percentile(coefs, 2.5, axis=0)),
    "IC95 bootstrap sup": np.exp(np.percentile(coefs, 97.5, axis=0)),
    "IC95 Wald inf": np.exp(wald[0]),
    "IC95 Wald sup": np.exp(wald[1]),
}, index=cols_mejor).drop(index="const")
tabla_or["Ancho bootstrap / Wald"] = ((tabla_or["IC95 bootstrap sup"] - tabla_or["IC95 bootstrap inf"]) /
                                      (tabla_or["IC95 Wald sup"] - tabla_or["IC95 Wald inf"]))
print(tabla_or.round(3))

In [ ]:
# CELDA 4: Evaluación en prueba con umbral de Youden

# 1. El umbral se elige con ENTRENAMIENTO, para que la prueba sea honesta
p_train = mejor_modelo.predict(X_train[cols_mejor])
fpr_tr, tpr_tr, umbrales = roc_curve(y_train, p_train)
umbral = umbrales[np.argmax(tpr_tr - fpr_tr)]

# 2. Se aplica a PRUEBA
p_test = mejor_modelo.predict(X_test[cols_mejor])
y_pred = (p_test >= umbral).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

metricas = pd.Series({
    "AUC prueba": roc_auc_score(y_test, p_test),
    "AUC entrenamiento": roc_auc_score(y_train, p_train),
    "Umbral Youden": umbral,
    "Sensibilidad": tp / (tp + fn),
    "Especificidad": tn / (tn + fp),
    "Exactitud": accuracy_score(y_test, y_pred),
}).round(4)
print(metricas)

print("\nMatriz de confusión (prueba):")
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=["Real: no atraso alto", "Real: atraso alto"],
                   columns=["Predice: no", "Predice: sí"]))
print(classification_report(y_test, y_pred))

# 3. Curva ROC
fpr, tpr, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color="#d95f02", label=f"{nombre_mejor} (AUC = {metricas['AUC prueba']:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color="#999999", lw=1)
ax.scatter(1 - metricas["Especificidad"], metricas["Sensibilidad"], color="#333333", zorder=3,
           label=f"Umbral Youden = {umbral:.3f}")
ax.set_xlabel("1 − especificidad")
ax.set_ylabel("Sensibilidad")
ax.set_title("Curva ROC en el conjunto de prueba")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("../informe/figuras/s3_04_roc.png", dpi=150)
plt.show()

In [ ]:
# CELDA 5: Guardar resultados para la parte 3
tabla_modelos.to_csv("../resultados/logistica_modelos.csv")
tabla_or.round(4).to_csv("../resultados/logistica_odds_ratios.csv", index_label="Variable")
metricas.to_frame("valor").to_csv("../resultados/logistica_metricas.csv", index_label="Métrica")
print("Guardados en S3/resultados/: logistica_modelos.csv, logistica_odds_ratios.csv, logistica_metricas.csv")